# Project: AlphaZero-lite tic-tac-toe 🧠🎮
AlphaZero learned Go, chess and shogi from **self-play only**: a neural network proposes moves (policy) and judges positions (value), Monte Carlo tree search uses the network to look ahead, and the search results become the network's next training targets. Here is the whole loop on tic-tac-toe, small enough for a laptop CPU, from random play to never losing against random players and drawing against a perfect one.

Topic: [[Q-Learning and Policy Gradients]], [[RL Basics and MDPs]] · guide note: [[Project - AlphaZero-Lite Tic-Tac-Toe]] · paper: [Silver et al. 2017 – AlphaZero](https://arxiv.org/abs/1712.01815)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, random, functools, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); random.seed(0); np.random.seed(0); torch.set_num_threads(1)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]
def winner(b):        # b: tuple of 9 in {1, -1, 0}; returns 1/-1 for a win, 0 for draw, None if ongoing
    for i, j, k in LINES:
        if b[i] != 0 and b[i] == b[j] == b[k]: return b[i]
    return 0 if 0 not in b else None
def legal(b): return [i for i in range(9) if b[i] == 0]
def play(b, a, player): b = list(b); b[a] = player; return tuple(b)
@functools.lru_cache(None)
def perfect_value(b, player):   # exact minimax (tic-tac-toe is tiny), from `player`'s point of view
    w = winner(b)
    if w is not None: return w * player
    return max(-perfect_value(play(b, a, player), -player) for a in legal(b))
def perfect_move(b, player): return max(legal(b), key=lambda a: -perfect_value(play(b, a, player), -player))

class Net(nn.Module):           # input: board from the current player's view (+1 me, -1 opponent); outputs move logits and a value in [-1, 1]
    def __init__(self):
        super().__init__(); self.body = nn.Sequential(nn.Linear(9, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU())
        self.pi, self.v = nn.Linear(64, 9), nn.Sequential(nn.Linear(64, 1), nn.Tanh())
    def forward(self, x): h = self.body(x); return self.pi(h), self.v(h)
class Node:
    def __init__(self, priors): self.P = priors; self.N = {a: 0 for a in priors}; self.W = {a: 0.0 for a in priors}; self.children = {}

## 1. PUCT: which move to explore?
AlphaZero's selection rule balances the **average value** of a move, $Q(a)=W(a)/N(a)$, with an exploration bonus that's large for moves the network likes ($P(a)$) and that haven't been tried much:
$$a^* = \arg\max_a\ Q(a) + c_{\text{puct}}\,P(a)\,\frac{\sqrt{\sum_b N(b) + 1}}{1+N(a)}$$
(use $Q=0$ for unvisited moves). `node.P`, `node.N`, `node.W` are dicts over legal moves.

In [ ]:
def puct_select(node, c_puct=1.5):
    # TODO 1: argmax of Q + U over the node's moves
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_puct():
    n = Node({0: 0.5, 1: 0.5}); n.N = {0: 10, 1: 0}; n.W = {0: 5.0, 1: 0.0}
    m = Node({0: 0.9, 1: 0.1}); m.N = {0: 1, 1: 1}; m.W = {0: -1.0, 1: 1.0}
    return puct_select(n) == 1 and puct_select(Node({3: 0.1, 5: 0.8, 7: 0.1})) == 5 and puct_select(m, c_puct=0.1) == 1
check("PUCT", _t_puct, "Unvisited moves get explored; with no visits the prior decides; with tiny c the better Q wins.")

def mcts_policy(net, board, player, sims=60, c_puct=1.5, noise=True):
    """Run MCTS from `board` (player to move); return visit-count distribution over 9 moves."""
    def evaluate(b, p):
        with torch.no_grad(): logits, v = net(torch.tensor([x * p for x in b], dtype=torch.float32))
        moves = legal(b); pr = torch.softmax(logits[moves], 0).numpy()
        return dict(zip(moves, pr)), float(v)
    pri, _ = evaluate(board, player)
    if noise:                                                         # Dirichlet noise at the root for exploration (as in AlphaZero)
        d = np.random.dirichlet([0.5] * len(pri)); pri = {a: 0.75 * p + 0.25 * x for (a, p), x in zip(pri.items(), d)}
    root = Node(pri)
    for _ in range(sims):
        node, b, p, path = root, board, player, []
        while True:
            a = puct_select(node, c_puct); path.append((node, a)); b = play(b, a, p); p = -p
            w = winner(b)
            if w is not None: value = -abs(w); break                  # the player who just moved won (value for the side to move = −1) or draw (0)
            if a not in node.children:
                pri, value = evaluate(b, p); node.children[a] = Node(pri); break
            node = node.children[a]
        for nd, a in reversed(path):                                  # back up: value is from the perspective of the side to move after `a`
            value = -value; nd.N[a] += 1; nd.W[a] += value
    pi = np.zeros(9)
    for a, n in root.N.items(): pi[a] = n
    return pi / pi.sum()

## 2. The AlphaZero loss\nTrain the network to predict the **search's** move distribution $\pi$ (cross-entropy with the policy logits) and the **game result** $z\in\{-1,0,1\}$ from the current player's view (MSE with the value head): $\ell = -\pi^\top\log p + (v-z)^2$.

In [ ]:
def az_loss(logits, value, target_pi, target_z):
    # TODO 2: policy cross-entropy (soft targets) + value MSE
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_azl():
    logits = torch.zeros(2, 9); v = torch.tensor([[0.5], [-0.5]]); pi = torch.full((2, 9), 1 / 9); z = torch.tensor([1.0, -1.0])
    return abs(az_loss(logits, v, pi, z).item() - (math.log(9) + 0.25)) < 1e-5
check("AlphaZero loss", _t_azl, "Uniform logits vs uniform π give ln 9; values 0.5/−0.5 vs targets 1/−1 give MSE 0.25.")

def self_play_game(net):
    board, player, hist = (0,) * 9, 1, []
    while winner(board) is None:
        pi = mcts_policy(net, board, player); hist.append(([x * player for x in board], pi, player))
        a = int(np.random.choice(9, p=pi)) if len(hist) < 3 else int(np.argmax(pi))      # explore the opening, then play the best move
        board = play(board, a, player); player = -player
    w = winner(board); return [(s, pi, w * p) for s, pi, p in hist]

def arena(net, opponent, games=40):
    score = {"win": 0, "draw": 0, "loss": 0}
    for g in range(games):
        board, player, net_side = (0,) * 9, 1, 1 if g % 2 == 0 else -1
        while winner(board) is None:
            a = int(np.argmax(mcts_policy(net, board, player, noise=False))) if player == net_side else opponent(board, player)
            board = play(board, a, player); player = -player
        w = winner(board); score["win" if w == net_side else "draw" if w == 0 else "loss"] += 1
    return score

if ready("PUCT", "AlphaZero loss"):
    net = Net(); opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=1e-4); buffer = []
    rand = lambda b, p: random.choice(legal(b)); perfect = lambda b, p: perfect_move(b, p)
    print("before training vs perfect player:", arena(net, perfect, 20))
    for it in range(12):
        for _ in range(25): buffer += self_play_game(net)
        buffer = buffer[-3000:]
        for _ in range(150):
            batch = random.sample(buffer, min(64, len(buffer)))
            s = torch.tensor([x[0] for x in batch], dtype=torch.float32); pi = torch.tensor(np.array([x[1] for x in batch]), dtype=torch.float32); z = torch.tensor([float(x[2]) for x in batch])
            logits, v = net(s); loss = az_loss(logits, v, pi, z); opt.zero_grad(); loss.backward(); opt.step()
        if it % 3 == 2: print(f"iteration {it + 1}: loss {loss.item():.3f}")
    vs_random, vs_perfect = arena(net, rand, 40), arena(net, perfect, 20)
    print("vs random player:", vs_random, "| vs perfect player:", vs_perfect)
    check("learned by self-play", lambda: vs_random["loss"] == 0 and vs_perfect["loss"] <= 2,
          "After self-play it should never lose to a random player and (almost) always draw against perfect play.")

<details><summary>▶ Solution</summary>

```python
def puct_select(node, c_puct=1.5):
    total = sum(node.N.values())
    best, best_score = None, -1e9
    for a in node.P:
        q = node.W[a] / node.N[a] if node.N[a] > 0 else 0.0
        u = c_puct * node.P[a] * math.sqrt(total + 1) / (1 + node.N[a])
        if q + u > best_score:
            best, best_score = a, q + u
    return best
```

```python
def az_loss(logits, value, target_pi, target_z):
    return F.cross_entropy(logits, target_pi) + F.mse_loss(value.squeeze(-1), target_z)
```
</details>

## Stretch goals
- Use the *raw network* (argmax of the policy head, no search) as a player. How strong is it alone? How much does search add?
- Scale up to Connect Four ([[Fun Projects]] has the game): convolutional net, more simulations, more patience.
- Plot the value head's prediction for the empty board over training. It should go to 0: perfect tic-tac-toe is a draw.